In [1]:
dashboard_code = r'''
"""
EcoEcon Live Dashboard
Ecological Resilience Indicators for Systemic Risk
Run with: streamlit run dashboard.py
"""

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
import os
import sys

sys.path.insert(0, os.path.expanduser('~/ecoecon'))

st.set_page_config(
    page_title="EcoEcon — Ecological Risk Indicators",
    page_icon="🌿",
    layout="wide"
)

st.title("🌿 EcoEcon — Ecological Resilience Indicators for Systemic Risk")
st.caption(
    "Modeling the US economy as a biological ecosystem. "
    "Sectors = species. Supply chains = mutualistic dependencies. "
    "Crises = extinction cascades. | Abhinav Vaddi"
)

# ── Load data ──────────────────────────────────────────────────
@st.cache_data
def load_data():
    data_dir = os.path.expanduser('~/ecoecon/data/processed')
    master_2008  = pd.read_csv(os.path.join(data_dir, 'master_2008.csv'))
    master_2020  = pd.read_csv(os.path.join(data_dir, 'master_2020.csv'))
    calm_postgfc = pd.read_csv(os.path.join(data_dir, 'master_calm_post_gfc_recovery.csv'))
    calm_precovid= pd.read_csv(os.path.join(data_dir, 'master_calm_pre_covid_expansion.csv'))
    csd_housing  = pd.read_csv(os.path.join(data_dir, 'csd_housing_starts.csv'),
                               parse_dates=['date'])
    model_results= pd.read_csv(os.path.join(data_dir, 'model_results_honest.csv'))
    robustness   = pd.read_csv(os.path.join(data_dir, 'robustness_battery.csv'))
    return (master_2008, master_2020, calm_postgfc, calm_precovid,
            csd_housing, model_results, robustness)

(master_2008, master_2020, calm_postgfc, calm_precovid,
 csd_housing, model_results, robustness) = load_data()

COLORS = {
    'ebi':        '#1D9E75',
    'spectral':   '#534AB7',
    'housing':    '#D85A30',
    'crisis':     '#D85A30',
    'negative':   '#534AB7',
    'gray':       '#888780',
}

# ── Sidebar ────────────────────────────────────────────────────
st.sidebar.header("Key Results")

st.sidebar.metric(
    "Spectral gap Kendall τ (2002–2007)",
    "-0.87",
    help="Monotonic pre-crisis decline. Negative = fragmentation increasing."
)
st.sidebar.metric(
    "LOO AUC (Logistic Regression)",
    "1.000",
    help="Leave-one-out cross-validation. p=0.034 permutation test. n=7."
)
st.sidebar.metric(
    "False alarms in calm periods",
    "0 / 10",
    help="Spectral gap produces zero false alarms across 2010–2019."
)
st.sidebar.metric(
    "2020 negative control (LR)",
    "0.002–0.004",
    help="Crisis probability assigned to 2015–2019 by model trained on 2008."
)

st.sidebar.divider()
st.sidebar.warning(
    "This is a retrospective analysis of one positive crisis episode. "
    "Results do not constitute a real-time prediction system. "
    "Multi-crisis validation is required before any deployment claim."
)
st.sidebar.caption(f"Updated: {datetime.now().strftime('%Y-%m-%d')}")

# ── Tabs ───────────────────────────────────────────────────────
tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "📊 Signal Analysis",
    "🔄 Negative Control",
    "🚨 False Positive Analysis",
    "🤖 Model Performance",
    "📖 Methodology"
])

# ── Tab 1: Signal Analysis ─────────────────────────────────────
with tab1:
    st.subheader("Ecological Signals — 2008 GFC Episode")
    st.caption(
        "Two of five signals behaved as theorized. "
        "Honest verdicts shown for all five."
    )

    signals = [
        ('spectral_gap',       'Spectral gap',           '✓ Declining 2002–2007 (τ=−0.87)',  COLORS['spectral'], True),
        ('ebi',                'EBI',                    '✗ Wrong direction pre-2008',        COLORS['ebi'],      False),
        ('Housing starts_var', 'Housing starts variance','✓ Peaks ~18mo before crisis',       COLORS['housing'],  True),
        ('Housing starts_ar1', 'Housing starts AR(1)',   '✗ Wrong direction pre-2008',        COLORS['gray'],     False),
        ('Credit spread_var',  'TED spread variance',    '~ Coincident, not leading',         '#BA7517',          False),
    ]

    fig, axes = plt.subplots(5, 1, figsize=(12, 16))
    crisis_start = 2007
    crisis_end   = 2009

    for ax, (col, title, verdict, color, works) in zip(axes, signals):
        if col not in master_2008.columns:
            continue
        years = master_2008['year'].tolist()
        vals  = master_2008[col].tolist()
        ax.plot(years, vals, color=color, linewidth=2.5,
                marker='o', markersize=7, zorder=3)
        ax.axvspan(crisis_start - 0.5, crisis_end + 0.5,
                   alpha=0.10, color=COLORS['crisis'])
        verdict_color = '#1D9E75' if '✓' in verdict else '#D85A30' if '✗' in verdict else '#BA7517'
        ax.text(0.02, 0.90, verdict, transform=ax.transAxes,
                fontsize=9, color=verdict_color, va='top',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                          edgecolor=verdict_color, alpha=0.8))
        ax.set_title(title, fontsize=10)
        ax.set_xticks(years)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.suptitle('Ecological early warning signals — 2008 GFC\n'
                 'Red shading = crisis period',
                 fontsize=12, fontweight='normal')
    plt.tight_layout()
    st.pyplot(fig)

    st.info(
        "**What works:** Spectral gap (network fragmentation) and housing "
        "starts variance (critical slowing down) both move in the theoretically "
        "predicted direction before the 2008 crisis. Three signals do not — "
        "this is reported honestly."
    )

# ── Tab 2: Negative Control ────────────────────────────────────
with tab2:
    st.subheader("2020 COVID — Theoretically Motivated Negative Control")
    st.markdown(
        "The 2020 COVID shock was an **exogenous perturbation**, not an endogenous "
        "approach to a bifurcation. Ecological theory predicts critical slowing down "
        "signals should **not** appear before an exogenous shock. This is a "
        "discriminating test of the theory, not just a second positive example."
    )

    col1, col2 = st.columns(2)

    with col1:
        st.markdown("**Spectral gap**")
        fig, axes = plt.subplots(2, 1, figsize=(6, 8))
        for ax, (master, ep_label, cs, ce, color) in zip(axes, [
            (master_2008, '2008 GFC\n(expect signal)',    2007, 2009, COLORS['crisis']),
            (master_2020, '2020 COVID\n(expect NO signal)', 2020, 2021, COLORS['negative']),
        ]):
            years = master['year'].tolist()
            vals  = master['spectral_gap'].tolist()
            ax.plot(years, vals, color=color, linewidth=2.5,
                    marker='o', markersize=7)
            ax.axvspan(cs - 0.5, ce + 0.5, alpha=0.10, color=COLORS['crisis'])
            ax.set_title(ep_label, fontsize=10)
            ax.set_xticks(years)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
        plt.tight_layout()
        st.pyplot(fig)

    with col2:
        st.markdown("**Housing starts variance**")
        fig, axes = plt.subplots(2, 1, figsize=(6, 8))
        for ax, (master, ep_label, cs, ce, color) in zip(axes, [
            (master_2008, '2008 GFC\n(expect signal)',    2007, 2009, COLORS['crisis']),
            (master_2020, '2020 COVID\n(expect NO signal)', 2020, 2021, COLORS['negative']),
        ]):
            if 'Housing starts_var' not in master.columns:
                continue
            years = master['year'].tolist()
            vals  = master['Housing starts_var'].tolist()
            ax.plot(years, vals, color=color, linewidth=2.5,
                    marker='o', markersize=7)
            ax.axvspan(cs - 0.5, ce + 0.5, alpha=0.10, color=COLORS['crisis'])
            ax.set_title(ep_label, fontsize=10)
            ax.set_xticks(years)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
        plt.tight_layout()
        st.pyplot(fig)

    st.success(
        "**Negative control holds.** Spectral gap shows no directional trend "
        "pre-COVID (τ=+0.20 vs τ=−0.87 before 2008). Housing variance was "
        "declining 2015–2019 — the opposite of the pre-2008 buildup. "
        "Four of five signals show no pre-crisis buildup before the exogenous shock."
    )

# ── Tab 3: False Positive Analysis ────────────────────────────
with tab3:
    st.subheader("False Positive Analysis — Does the Signal Fire During Calm Periods?")
    st.markdown(
        "Every early warning system must answer: does it fire when nothing is happening? "
        "We test spectral gap across ten calm-period years (2010–2019)."
    )

    fig, ax = plt.subplots(figsize=(13, 5))

    net_2008 = pd.read_csv(
        os.path.expanduser('~/ecoecon/data/processed/network_metrics_2008.csv')
    )
    net_2020 = pd.read_csv(
        os.path.expanduser('~/ecoecon/data/processed/network_metrics_2020.csv')
    )
    calm_postgfc_net = pd.read_csv(
        os.path.expanduser('~/ecoecon/data/processed/network_metrics_calm_post_gfc_recovery.csv')
    )
    calm_precovid_net = pd.read_csv(
        os.path.expanduser('~/ecoecon/data/processed/network_metrics_calm_pre_covid_expansion.csv')
    )

    ax.plot(net_2008['year'], net_2008['spectral_gap'],
            color=COLORS['crisis'], linewidth=2.5, marker='o',
            markersize=8, label='2008 GFC episode', zorder=4)
    ax.plot(net_2020['year'], net_2020['spectral_gap'],
            color=COLORS['negative'], linewidth=2.5, marker='s',
            markersize=8, label='2020 COVID episode', zorder=4, linestyle='--')
    ax.plot(calm_postgfc_net['year'], calm_postgfc_net['spectral_gap'],
            color=COLORS['ebi'], linewidth=1.5, marker='^',
            markersize=7, label='Calm: 2010–2014', alpha=0.7, linestyle=':')
    ax.plot(calm_precovid_net['year'], calm_precovid_net['spectral_gap'],
            color=COLORS['gray'], linewidth=1.5, marker='v',
            markersize=7, label='Calm: 2015–2019', alpha=0.7, linestyle=':')

    ax.set_ylabel('Spectral gap (technical coefficients)')
    ax.set_xlabel('Year')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    st.pyplot(fig)

    col1, col2, col3 = st.columns(3)
    col1.metric("False alarms 2010–2014", "0 / 5")
    col2.metric("False alarms 2015–2019", "0 / 5")
    col3.metric("Total false alarm rate", "0%", delta="10 calm years tested")

    st.success(
        "Spectral gap produces **zero false alarms** across ten calm-period years. "
        "It is not merely tracking the business cycle."
    )

# ── Tab 4: Model Performance ───────────────────────────────────
with tab4:
    st.subheader("Model Performance — Honest LOO Evaluation")
    st.warning(
        "All results use leave-one-out cross-validation on n=7 annual samples. "
        "Wide confidence intervals are expected and acknowledged. "
        "Multi-crisis validation is required before strong predictive claims."
    )

    col1, col2 = st.columns(2)

    with col1:
        st.markdown("**LOO AUC by model**")
        fig, ax = plt.subplots(figsize=(6, 4))
        models = ['Logistic\nRegression', 'Random\nForest']
        aucs   = [1.000, 0.400]
        colors = [COLORS['ebi'], COLORS['spectral']]
        bars = ax.bar(models, aucs, color=colors, alpha=0.85, width=0.4)
        ax.axhline(0.5, color=COLORS['gray'], linewidth=1,
                   linestyle='--', label='Random chance')
        for bar, auc in zip(bars, aucs):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.02,
                    f'{auc:.3f}', ha='center', fontsize=11,
                    fontweight='bold')
        ax.set_ylim(0, 1.2)
        ax.set_ylabel('LOO ROC-AUC')
        ax.legend(fontsize=9)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        st.pyplot(fig)

    with col2:
        st.markdown("**Key numbers**")
        st.metric("LR LOO AUC", "1.000")
        st.metric("Permutation test p-value", "0.034")
        st.metric("2007 advance warning (LR)", "0.013", help="Did not fire — honest null result")
        st.metric("2020 pre-crisis probability (LR)", "0.002–0.004")

    st.markdown("**Robustness battery**")
    st.dataframe(
        robustness[['test', 'LR_LOO_AUC', 'Kendall_tau', 'p_value']],
        use_container_width=True,
        hide_index=True
    )

    st.info(
        "**Random Forest underperforms** (AUC=0.400), consistent with overfitting "
        "at small sample sizes. Linear models are preferred at n=7. "
        "CSD signals alone achieve AUC=1.000; network metrics are not required "
        "for classification at this scale."
    )

# ── Tab 5: Methodology ─────────────────────────────────────────
with tab5:
    st.subheader("Methodology")

    st.markdown("### Variable Mapping")
    st.markdown("""
| Ecological Concept | Economic Analog | Note |
|---|---|---|
| Species | Sector (68 BEA industries) | — |
| Mutualistic dependency | Supplier-buyer relationship | Corrected from predator-prey |
| Biomass flow | Revenue flow (technical coefficients) | Column-normalized, not raw flows |
| Keystone species | Systemically important sector | Finance identified pre-2008 |
| Extinction cascade | Bankruptcy contagion | — |
| Ecosystem biodiversity | Sector centrality distribution (EBI) | Did not work as theorized |
| Critical slowing down | Rising variance + autocorrelation | Works for housing starts |
| Spectral gap collapse | Network fragmentation | Strongest ecological signal |
    """)

    st.markdown("### Key Methodological Decisions")
    st.markdown("""
- **Technical coefficients** (not raw dollar flows) for network normalization — removes nominal GDP/inflation bias
- **RobustScaler fitted on pre-crisis years only** — prevents look-ahead leakage
- **Leave-one-out cross-validation** — honest evaluation on n=7
- **2007 held out as advance warning test** — not labeled, tested separately
- **Episode types pre-registered** before analysis (2008 = endogenous, 2020 = exogenous)
- **Cubic spline interpolation noted as leaky** — monthly network features are retrospective only
    """)

    st.markdown("### Honest Limitations")
    st.error("""
**Single positive crisis episode.** All metrics have wide confidence intervals.
Multi-crisis leave-one-out validation is required before generalizability claims.

**Wrong network for a financial crisis.** BEA IO tables capture supply-chain flows,
not financial exposure networks. Future work needs FFIEC Call Report data.

**Three signals failed.** EBI, housing AR(1), and TED spread variance did not
behave as theorized. This constrains which ecological mechanisms are supported.

**No real-time validation.** Signals computed on revised data vintages.
ALFRED real-time data needed for genuine out-of-sample claims.
    """)

    st.markdown("### References")
    st.markdown("""
1. May, R.M. (1972). Will a large complex system be stable? *Nature*, 238, 413–414.
2. Scheffer et al. (2009). Early-warning signals for critical transitions. *Nature*, 461, 53–59.
3. Acemoglu et al. (2015). Systemic risk and stability in financial networks. *AER*, 105(2).
4. Haldane & May (2011). Systemic risk in banking ecosystems. *Nature*, 469, 351–355.
5. Allesina & Tang (2012). Stability criteria for complex ecosystems. *Nature*, 483, 205–208.
6. Dakos et al. (2012). Methods for detecting early warnings. *PLOS ONE*, 7(7).
    """)

st.divider()
st.caption(
    "🌿 EcoEcon — Ecological Resilience Indicators for Systemic Risk | "
    "Abhinav Vaddi | github.com/Abhiv1028/EcoEcon | "
    "Retrospective analysis only — not a real-time prediction system"
)
'''

In [ ]:
import subprocess
import os

dashboard_path = os.path.expanduser('~/ecoecon/src/dashboard.py')

print("Launching EcoEcon Dashboard...")
print(f"File: {dashboard_path}")
print()
print("The dashboard will open in your browser.")
print("Press Ctrl+C in the terminal to stop it.")
print()

# This will start Streamlit — it'll take over the cell output
!streamlit run {dashboard_path} --server.port 8501

Launching EcoEcon Dashboard...
File: /Users/abhinavvaddi/ecoecon/src/dashboard.py

The dashboard will open in your browser.
Press Ctrl+C in the terminal to stop it.


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://10.0.0.106:8501

2026-07-04 00:02:32.661 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
